# 01 — Baseline Exploration

**Purpose:** Understand the data before touching any model. Every ML project starts here — you must know your data's shape, quirks, and distributions before you can make good decisions about features and models.

**Run from project root:**
```bash
PYTHONPATH=. jupyter notebook ml/notebooks/01_baseline.ipynb
```

**What we'll do in this notebook:**
1. Connect to the DB and load `derived.product_daily_features`
2. Check shape, dtypes, and null counts — know what's missing
3. Understand the distribution of `quantity_sold` — is it skewed? Are there outliers?
4. Count `stockout_proxy` rows — these are excluded from training
5. Find products with sparse history (< 30 days of sales) — model will struggle on these
6. Compute WMA baseline accuracy — this is the bar our model must beat

---

## Cell 1 — Imports and DB connection

We use SQLAlchemy to connect (same pattern as the rest of the project). `pandas.read_sql` pulls the full table into a DataFrame.

**Why load the whole table?** At ~4.6M rows it fits comfortably in RAM (~500MB). Loading once and working in-memory is faster than repeated DB queries.

In [ ]:
import os
import sys
from pathlib import Path
from urllib.parse import quote_plus

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Load .env from project root
load_dotenv(Path.cwd().parent.parent / ".env")  # adjust if notebook is run from a different cwd

password = quote_plus(os.getenv("password", ""))
DATABASE_URL = (
    f"postgresql+psycopg2://{os.getenv('user')}:{password}"
    f"@{os.getenv('host')}:{os.getenv('port')}/{os.getenv('dbname')}"
)
engine = create_engine(DATABASE_URL, future=True, pool_pre_ping=True)
print("Connected:", engine.url)

## Cell 2 — Load `derived.product_daily_features`

This is the dense table: every (date × product) pair from the first sale date to today, including zero-sale days.

**Columns we care about:**
- `quantity_sold` — what the model will predict
- `lag_1_qty`, `lag_7_qty` — yesterday and same weekday last week
- `last_7/30/60_day_avg` — rolling averages
- `last_7_day_stddev` — volatility
- `day_of_week` — 0=Monday, 6=Sunday
- `stockout_proxy` — True if zero-sale day was probably due to empty shelf (exclude from training)
- `is_holiday`, `days_to_next_festival`, `days_since_last_festival` — calendar signals (may be NULL)

In [ ]:
# Load the full features table — this may take 30–60 seconds on first run
df = pd.read_sql(
    text("""
        SELECT
            date, product_id, quantity_sold,
            lag_1_qty, lag_7_qty,
            last_7_day_avg, last_30_day_avg, last_60_day_avg,
            last_7_day_stddev, day_of_week,
            is_holiday, days_to_next_festival, days_since_last_festival,
            stockout_proxy
        FROM derived.product_daily_features
        ORDER BY product_id, date
    """),
    con=engine,
    parse_dates=["date"],
)
print(f"Rows: {len(df):,}  |  Products: {df['product_id'].nunique():,}  |  Date range: {df['date'].min()} → {df['date'].max()}")

## Cell 3 — Shape, dtypes, null counts

Always do this first. Nulls in feature columns won't crash XGBoost (it handles them natively), but knowing which columns are sparse helps you understand what the model is working with.

**What to look for:**
- `lag_1_qty` should have ~1 null per product (first day has no yesterday)
- `is_holiday`, `days_to_next_festival` may be 100% null if `calendar_dim` isn't seeded — that's OK
- `last_7_day_stddev` may be null for first 6 days per product — expected

In [ ]:
print(df.dtypes)
print("\nNull counts:")
print(df.isnull().sum())
print("\nNull %:")
print((df.isnull().sum() / len(df) * 100).round(2))

## Cell 4 — Distribution of `quantity_sold`

Retail demand is almost always **right-skewed**: most products sell 0–5 units/day, a few sell 50+. This matters because XGBoost (like most tree models) handles skewed targets reasonably — unlike linear regression which assumes normality.

**What to look for:**
- What % of rows are zero-sale days? If >50%, the model needs to handle sparsity
- Are there extreme outliers (e.g., 500 units on a single day)? Those are festival/promotion days
- What's the 99th percentile? That's your rough sanity cap threshold

In [ ]:
print(df["quantity_sold"].describe())
print(f"\nZero-sale rows: {(df['quantity_sold'] == 0).sum():,} ({(df['quantity_sold'] == 0).mean()*100:.1f}%)")
print(f"99th percentile: {df['quantity_sold'].quantile(0.99):.1f}")
print(f"Max: {df['quantity_sold'].max():.1f}")

# Histogram (clip at 50 so sparse products don't dominate the x-axis)
df[df["quantity_sold"] <= 50]["quantity_sold"].hist(bins=50, figsize=(10, 4))
plt.xlabel("quantity_sold")
plt.ylabel("row count")
plt.title("Distribution of quantity_sold (capped at 50)")
plt.show()

## Cell 5 — Count `stockout_proxy` rows

`stockout_proxy = True` means: the product had recent sales activity but sold zero today and received no stock. These are likely empty-shelf days — the customer demand existed but couldn't be fulfilled.

**Why exclude them from training?** If we train the model on these zeros, it learns 'low demand' for days that actually had high demand. That's a biased signal. We exclude them; the model never sees them.

In [ ]:
n_stockout = df["stockout_proxy"].sum()
pct = n_stockout / len(df) * 100
print(f"Stockout proxy rows: {n_stockout:,} ({pct:.2f}% of all rows)")
print(f"Rows remaining after exclusion: {len(df) - n_stockout:,}")

## Cell 6 — Products with sparse history

Products with fewer than 30 days of actual sales (non-zero, non-stockout) have too little history for the model to learn from. They'll still get predictions, but those predictions will be weaker. It's good to know how many there are.

**Why 30 days?** That's roughly the minimum needed for a 30-day rolling average to be meaningful. Products with less than 30 days will have null `last_30_day_avg` — XGBoost handles this but the prediction quality is lower.

In [ ]:
active_days = (
    df[(df["quantity_sold"] > 0) & (~df["stockout_proxy"])]
    .groupby("product_id")["date"]
    .nunique()
    .reset_index()
    .rename(columns={"date": "active_days"})
)
sparse = active_days[active_days["active_days"] < 30]
print(f"Products with < 30 active sale days: {len(sparse):,} ({len(sparse)/active_days['product_id'].nunique()*100:.1f}%)")
print(active_days["active_days"].describe())

## Cell 7 — WMA baseline accuracy (the bar to beat)

Before training any model, we compute the accuracy of the current system: the Weighted Moving Average.

**Formula:** `0.6 × last_7_day_avg + 0.3 × last_30_day_avg + 0.1 × last_60_day_avg`

We evaluate on the **last 30 days** of the dataset (excluding stockout rows). This is the same hold-out period the trained model will be evaluated on — apples-to-apples comparison.

**MAE** = Mean Absolute Error = "on average, WMA is off by X units per product per day"

Write down the WMA MAE — we need it later to measure improvement.

In [ ]:
from sklearn.metrics import mean_absolute_error

# Hold-out: last 30 days, exclude stockout rows, exclude rows with null lag features
cutoff = df["date"].max() - pd.Timedelta(days=30)
test = df[
    (df["date"] > cutoff) &
    (~df["stockout_proxy"]) &
    (df["lag_1_qty"].notna())
].copy()

test["wma_pred"] = (
    0.6 * test["last_7_day_avg"].fillna(0)
    + 0.3 * test["last_30_day_avg"].fillna(0)
    + 0.1 * test["last_60_day_avg"].fillna(0)
)

wma_mae = mean_absolute_error(test["quantity_sold"], test["wma_pred"])
wma_rmse = float(np.sqrt(((test["quantity_sold"] - test["wma_pred"]) ** 2).mean()))

print(f"WMA baseline — MAE: {wma_mae:.4f}  |  RMSE: {wma_rmse:.4f}")
print(f"Evaluated on {len(test):,} rows across {test['product_id'].nunique():,} products")
print(f"\n>>> Write this down: WMA MAE = {wma_mae:.4f} <<<")